# 02 · Flujo de compra end-to-end

Recorre login → catálogo → carrito → checkout → confirmación, validando los montos en cada paso.
La versión automatizada está en `tests/test_checkout.py`.

In [ ]:
import sys
from pathlib import Path

# Permite importar pages/ y utils/ desde la raíz del proyecto
sys.path.insert(0, str(Path.cwd().parent))

from IPython.display import Image, display

from pages import LoginPage, InventoryPage
from utils.config import USERS, PASSWORD
from utils.driver_factory import create_driver

def captura(page, width=700):
    display(Image(page.screenshot_png(), width=width))

## 1. Login

In [ ]:
driver = create_driver()
inventory = LoginPage(driver).load().login_as(USERS['standard'])
captura(inventory)

## 2. Explorar el catálogo

Listamos los productos ordenados por precio de menor a mayor.

In [ ]:
inventory.sort_by(InventoryPage.SORT_PRICE_ASC)
for nombre, precio in zip(inventory.product_names(), inventory.product_prices()):
    print(f'${precio:>6.2f}  {nombre}')

## 3. Agregar productos al carrito

In [ ]:
productos = ['Sauce Labs Backpack', 'Sauce Labs Fleece Jacket']
inventory.add_to_cart(*productos)
print('Productos en el carrito (badge):', inventory.cart_count())
captura(inventory)

In [ ]:
cart = inventory.go_to_cart()
print('En el carrito:', cart.item_names())
captura(cart)

## 4. Checkout: datos del comprador

In [ ]:
info = cart.checkout()
overview = info.continue_to_overview('Ricardo', 'Pavez', '8320000')
captura(overview)

## 5. Validar montos del resumen

In [ ]:
subtotal, impuesto, total = overview.subtotal(), overview.tax(), overview.total()
suma_precios = sum(overview.item_prices())

print(f'Suma de precios: ${suma_precios:.2f}')
print(f'Subtotal:        ${subtotal:.2f}')
print(f'Impuesto:        ${impuesto:.2f}')
print(f'Total:           ${total:.2f}')

assert round(suma_precios, 2) == subtotal, 'El subtotal no coincide con la suma de precios'
assert round(subtotal + impuesto, 2) == total, 'El total no es subtotal + impuesto'
print('\nMontos OK')

## 6. Finalizar la compra

In [ ]:
complete = overview.finish()
print(complete.header())
captura(complete)

## Cierre

In [ ]:
driver.quit()